In [1]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

In [2]:
import os
import torch
from PIL import Image

from models.cnn_model import CNN_Model
from utils import get_device, build_cnn_transform,ResizeKeepRatioPad

# Parameters

In [17]:
ckpt_path = '../checkpoints/cnn_checkpoint.pt'
image_path = '../data/face/predict/'
formats = {".jpg", ".jpeg", ".png", ".bmp", ".gif", ".tiff", ".webp"}

# Import Predict data

In [18]:
resized_images = []

for i in os.listdir(image_path):

    path = image_path + '/' + i

    ext = os.path.splitext(i)[-1]

    if ext in formats:
        
        img = Image.open(path)
        resizer = ResizeKeepRatioPad( input_size )

        resized_img = resizer(img)
        resized_images.append( resized_img )       


# Load model and configs

In [19]:
device = get_device()

## Get saved training checkpoint

In [20]:
with open(ckpt_path, 'rb') as f:
    checkpoint = torch.load(f, map_location=device)

if not isinstance(checkpoint, dict):
    raise ValueError(
        "Checkpoint must be a dict with model_state_dict, model_config, preprocess_config, and class_names. "
        "Please retrain and save checkpoint_data in training."
    )

required_keys = {"model_state_dict", "model_config", "preprocess_config", "class_names"}
missing_keys = required_keys - set(checkpoint.keys())
if missing_keys:
    raise ValueError(
        f"Checkpoint is missing required keys: {sorted(missing_keys)}. "
        "Please retrain and save checkpoint_data in training."
    )

## get config and model from checkpoint

In [21]:
model_state_dict = checkpoint["model_state_dict"]
model_config = checkpoint["model_config"]
preprocess_config = checkpoint["preprocess_config"]
class_names = checkpoint["class_names"]
input_size = model_config['input_size']

## Build model and transformer

In [22]:
model = CNN_Model(**model_config).to(device)
transform = build_cnn_transform(preprocess_config)

In [23]:
model.load_state_dict(model_state_dict)
model.eval();

# Prediction

In [ ]:
predicted_labels = []

for item in resized_images:

    x = transform(item).unsqueeze(0).to(device)

    with torch.no_grad():
        logits = model(x)
        probs = torch.softmax(logits, dim=1)
        pred_idx = probs.argmax(dim=1).item()
        confidence = probs[0, pred_idx].item()

    pred_label = class_names[pred_idx]
    predicted_labels.append(pred_label)

    display(item)

    print(f"label: {pred_label}, confidence: {confidence:.4f}")

